# Fine-Tuning-Style Baseline And Limits

This notebook uses real provider calls with few-shot style examples, but no retrieved policy context. That makes the behavior-adaptation effect visible without running a real fine-tuning job.


## Learning Objectives

By the end of this notebook, you should be able to:

- Explain what supervised and parameter-efficient fine-tuning change.
- Use few-shot examples as a proxy for behavior adaptation.
- Separate tone imitation from the need for policy evidence.
- Explain the limits of the few-shot proxy.


## What This Proxy Can And Cannot Show

This notebook uses few-shot examples because they make behavior steering visible without running a training job. That is useful for strategy work: participants can see whether examples help tone, refusal style, or answer shape before deciding that real fine-tuning is worth the cost.

The proxy has hard limits. Few-shot examples are sent with one request and disappear afterward. They do not prove that a trained adapter would generalize, preserve safety behavior, or avoid regressions on unrelated tasks. Real fine-tuning still needs a held-out evaluation set and a rollback plan.


## Fine-Tuning Concepts Without Training Yet

**Supervised fine-tuning** teaches a model from examples of desired inputs and outputs. **Parameter-efficient fine-tuning** methods such as LoRA add or update a smaller set of parameters instead of changing every base-model weight. These approaches can improve repeated behavior, but they require careful evaluation because the model may also learn unwanted shortcuts.

In this strategy repo, we do not train. Instead, we use few-shot prompting as a low-cost probe. If examples do not help even in the prompt, real fine-tuning is unlikely to be the first thing to try. If examples help but the behavior is still inconsistent across held-out cases, that is evidence that a new fine-tuning run may be worth scoping with the workflow from the previous unit.


## What Real Fine-Tuning Changes

This notebook uses few-shot style examples as a live-call proxy for behavior adaptation. The figure below shows what real parameter-efficient fine-tuning changes instead, while the code here deliberately does not train weights or adapters.

Full fine-tuning keeps a separate copy of the model for each task. Prefix tuning freezes the pretrained model and trains only a small task-specific prefix. This is the same idea as the LoRA adapters from the previous unit: the new behavior is stored in a small set of trained parameters.

<div align="center">
  <img src="assets/04-prefix-tuning.png" alt="Full fine-tuning compared with prefix tuning" width="550" style="max-width: 100%; height: auto;" />
  <p><em>Full fine-tuning (one model copy per task) compared with prefix tuning (a small trained prefix on a frozen model). Source: <a href="https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/prefix-tuning.png" target="_blank" rel="noopener noreferrer">Hugging Face PEFT documentation</a></em></p>
</div>


In [ ]:
import json
from pprint import pprint

from pydantic import BaseModel, ValidationError

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    ReplySource,
    call_model,
    compact_style_examples,
    style_messages,
)

## Load Shared Data And Runtime Helpers

All notebooks use the same fixtures and provider gateway so the comparisons stay controlled. The repeated setup lives in [`strategy_helpers.py`](strategy_helpers.py); lesson cells can now focus on the architecture decision being tested. Dry mode remains deterministic when no provider key is available.


In [ ]:
print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

## Inspect Few-Shot Style Messages


In [ ]:
few_shot_messages = style_messages(STYLE_EXAMPLES)
print(f"Few-shot message count: {len(few_shot_messages)}")
print(compact_style_examples(STYLE_EXAMPLES))

## Call The Model Without Fresh Policy Context

The model receives style examples, but not policy documents. This intentionally tests the limitation: behavior can improve while factual grounding remains unproven.


## Reading Style-Only Outputs

Style-only outputs should be read with two questions in mind. First, did the answer follow the expected behavior pattern: concise, operational, careful refusal, or coaching? Second, did it claim policy facts that were not supplied in the prompt? A good style-only answer can still be incomplete because style examples do not carry source-of-truth policy evidence.

This is why the structured output includes `needs_policy_evidence`. That field is not a perfect judge, but it makes the limitation visible. When the model says it still needs policy evidence, it is admitting that behavior conditioning alone is not enough for grounded support answers.


## Examples Of Style-Only Success And Failure

A style-only answer succeeds when the task is mostly about response behavior. Example: “Reply in a concise operational tone and ask for the missing blocker.” The model can follow that pattern from examples.

A style-only answer fails when it needs current evidence. Example: “Can this duplicate payment be refunded?” The model may imitate the refund-response style, but it still needs the current refund policy to avoid inventing eligibility rules. This distinction is the reason strategy reviews separate behavior quality from grounding quality.


The style-only schema separates tone imitation from evidence needs. This is the central limitation of the notebook.


In [ ]:
class StyleOnlyAnswer(BaseModel):
    """Structured output for style-conditioned answers without retrieval."""

    answer: str
    predicted_tone: str
    needs_policy_evidence: bool
    response_source: ReplySource = "simulated"

Build the style-only prompt from examples and ask for structured JSON. Notice that no policy context is supplied.


Build the style-only prompt in a helper so it is clear that no policy context is included.


In [ ]:
def style_only_messages(question: str) -> list[dict[str, str]]:
    """Create a few-shot style prompt without retrieved evidence."""
    return [
        {
            "role": "system",
            "content": "Use the style of the examples. Return JSON with answer, predicted_tone, and needs_policy_evidence.",
        },
        *few_shot_messages,
        {"role": "user", "content": question},
    ]

The dry output intentionally says policy evidence is still needed.


In [ ]:
dry_style_only = json.dumps(
    {
        "answer": "Ask for the invoice number and both payment references in a concise support tone, but verify the current refund policy before giving exact eligibility details.",
        "predicted_tone": "concise",
        "needs_policy_evidence": True,
    }
)

Validate the structured style-only response before using it in comparisons.


In [ ]:
def style_only_answer(question: str) -> StyleOnlyAnswer:
    """Ask the provider to answer from style examples without policy context."""
    reply = call_model(
        style_only_messages(question),
        dry_response=dry_style_only,
        response_format={"type": "json_object"},
    )

    try:
        result = StyleOnlyAnswer.model_validate_json(reply.text)
        result.response_source = reply.source
        return result
    except ValidationError:
        return StyleOnlyAnswer(
            answer=reply.text,
            predicted_tone="unknown",
            needs_policy_evidence=True,
            response_source=reply.source,
        )

Run a few requests to show that style conditioning can be consistent while still needing policy evidence.


In [ ]:
for request in REQUESTS[:3]:
    print()
    print(request["request_id"], request["question"])
    pprint(style_only_answer(request["question"]).model_dump())

## Compare What This Can And Cannot Solve

A style-conditioned call can learn the response pattern, but it still cannot prove the current refund window, rate limit, or MFA rule unless those facts are present in the prompt or already reliably known.


In [ ]:
# Summarize tone and evidence needs across the held-out requests.
summary_rows = []
for request in REQUESTS:
    output = style_only_answer(request["question"])
    summary_rows.append(
        {
            "request_id": request["request_id"],
            "expected_tone": request["preferred_tone"],
            "predicted_tone": output.predicted_tone,
            "response_source": output.response_source,
            "needs_policy_evidence": output.needs_policy_evidence,
        }
    )
pprint(summary_rows)

### Exercise: Find Unjustified Confidence

Every request in `REQUESTS` has `fresh_fact_required` set to `True`, so none of them can be answered reliably without policy evidence. Using `summary_rows`, list the request IDs where the style-only model nevertheless reported `needs_policy_evidence` as `False`.


In [ ]:
# @TODO: Find the fresh-fact requests where the model claimed it needed no policy evidence.


<details>
<summary>Show Reference Solution</summary>

```python
fresh_fact_ids = {
    request["request_id"] for request in REQUESTS if request["fresh_fact_required"]
}
overconfident_ids = [
    row["request_id"]
    for row in summary_rows
    if row["request_id"] in fresh_fact_ids and not row["needs_policy_evidence"]
]
overconfident_ids
```

</details>


In live runs, the model often reports that it needs no policy evidence for questions that depend on current policy, such as the refund or rate-limit requests. Its self-assessment is not a reliable grounding check: a model without the policy cannot know what the policy says. In dry mode, and for fallback rows, the deterministic response always says evidence is needed, so the list is empty. Run the notebook with an API key to see the model's own judgment.


## Check Your Understanding

1. Do few-shot examples persist as model behavior after the request is finished?
2. Do LoRA or prefix tuning automatically provide fresh policy evidence at answer time?
3. A style-only answer sounds confident and follows the expected tone. Why can it still not be trusted for policy facts?

<details>
<summary>Show Reference Answers</summary>

1. No. Few-shot examples condition only the current request; they do not update model weights or adapters.
2. No. They can adapt model behavior, but fresh evidence still requires retrieval or another external context source.
3. Style examples teach tone and response shape, not the current refund, MFA, or rate-limit rules. Without retrieved policy, the model fills the gaps from general knowledge, so it can add plausible details that the policy does not contain.

</details>


## Further Reading

- [Does Fine-Tuning LLMs on New Knowledge Encourage Hallucinations?](https://arxiv.org/abs/2405.05904) - Gekhman et al., 2024
- [Few-Shot Parameter-Efficient Fine-Tuning is Better and Cheaper than In-Context Learning](https://arxiv.org/abs/2205.05638) - Liu et al., 2022
